In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc


# 1. CORE CONFIGURATION MATRIX
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
SEED = 42
DATA_DIR = "/content/drive/MyDrive/my_dataset/thermal"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Engine bound to hardware: {device}")


random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


pure_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. DATASET LOADING & 80/10/10 RANDOM SPLITTING

full_dataset = datasets.ImageFolder(root=DATA_DIR)
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)
total_count = len(full_dataset)


all_indices = list(range(total_count))
random.seed(SEED)
random.shuffle(all_indices)

# Calculate Split Thresholds for 80% Train / 10% Val / 10% Test
train_split = int(0.80 * total_count)
val_split = int(0.90 * total_count)

train_indices = all_indices[:train_split]
val_indices = all_indices[train_split:val_split]
test_indices = all_indices[val_split:]

class CustomSubset(Dataset):
    def __init__(self, subset, indices, transform):
        self.subset = subset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.subset[self.indices[idx]]
        return self.transform(x), y

train_dataset = CustomSubset(full_dataset, train_indices, pure_transform)
val_dataset = CustomSubset(full_dataset, val_indices, pure_transform)
test_dataset = CustomSubset(full_dataset, test_indices, pure_transform)


# 4. DATALOADERS

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Transformer Engine Configured: {total_count} files split randomly into 80/10/10 partitions.")
print(f" --> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")


# 5. MODEL INITIALIZATION & PARTIAL FREEZING

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

#Default all parameters to frozen
for param in model.parameters():
    param.requires_grad = False

#Unfreezing only the final Transformer block (Layer 11)
freeze_until_layer = 11
for i, layer in enumerate(model.encoder.layers):
    if i >= freeze_until_layer:
        for param in layer.parameters():
            param.requires_grad = True

for param in model.encoder.ln.parameters():
    param.requires_grad = True

# Overhauled Classification Head
model.heads.head = nn.Sequential(
    nn.Dropout(p=0.1),
    nn.Linear(in_features=768, out_features=NUM_CLASSES)
)

for param in model.heads.head.parameters():
    param.requires_grad = True

model.to(device)

# 6. OPTIMIZATION SETUP
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
trainable_params = filter(lambda p: p.requires_grad, model.parameters())

optimizer = optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)


# 7. OPTIMIZED ENGINE LOOP
results = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0

try:
    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct = 0.0, 0
        train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)

        for images, labels in train_loop:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_loop.set_postfix(loss=loss.item())

        scheduler.step()

        # Evaluation Block
        model.eval()
        val_loss, val_correct = 0.0, 0
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)

        with torch.no_grad():
            for images, labels in val_loop:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_loop.set_postfix(loss=loss.item())

        epoch_train_loss = train_loss / len(train_dataset)
        epoch_train_acc = train_correct / len(train_dataset)
        epoch_val_loss = val_loss / len(val_dataset)
        epoch_val_acc = val_correct / len(val_dataset)

        results["train_loss"].append(epoch_train_loss)
        results["train_acc"].append(epoch_train_acc)
        results["val_loss"].append(epoch_val_loss)
        results["val_acc"].append(epoch_val_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] -> "
              f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% || "
              f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), 'best_vit_model.pth')
            print(f" New Peak Checkpoint Saved! Performance: {best_val_acc*100:.2f}%")

except KeyboardInterrupt:
    print("\nTraining interrupted manually. Proceeding to evaluation with available metrics...")


# 8. RESULTS
epochs_range = range(1, len(results["train_loss"]) + 1)
if len(epochs_range) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    axes[0].plot(epochs_range, results["train_loss"], label='Train Loss', color='#b31b1b', linewidth=2)
    axes[0].plot(epochs_range, results["val_loss"], label='Val Loss', color='#2b6cb0', linewidth=2, linestyle='--')
    axes[0].set_title('Cross-Entropy Optimization Curve', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Loss Metric Value')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, linestyle=':', alpha=0.6)

    axes[1].plot(epochs_range, [acc * 100 for acc in results["train_acc"]], label='Train Acc', color='#2f855a', linewidth=2)
    axes[1].plot(epochs_range, [acc * 100 for acc in results["val_acc"]], label='Val Acc', color='#c53030', linewidth=2, linestyle='--')
    axes[1].axhline(y=85, color='gray', linestyle=':', label='85% Target Threshold')
    axes[1].set_title('Target Metric Progress (%)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Accuracy Progression')
    axes[1].legend(loc='lower right')
    axes[1].grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()


# 9. EVALUATION MATRICES
if os.path.exists('best_vit_model.pth'):
    print("Loading peak checkpoint for evaluation on isolated test data...")
    model.load_state_dict(torch.load('best_vit_model.pth', map_location=device))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating Test Set"):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print("\n===== UNBIASED TEST SET METRICS REPORT =====")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, cbar=True, annot_kws={"size": 11, "weight": "bold"})
plt.title('Unbiased Test Set Confusion Matrix', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Predicted Emotion Label', fontsize=12, labelpad=10)
plt.ylabel('True Ground-Truth Label', fontsize=12, labelpad=10)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Individual Class ROC Curves
labels_onehot = np.eye(NUM_CLASSES)[all_labels]
rows = (NUM_CLASSES + 2) // 3
fig, axes = plt.subplots(rows, 3, figsize=(15, rows * 5))
axes = axes.ravel()

for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(labels_onehot[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    axes[i].plot(fpr, tpr, color='#2b6cb0', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
    axes[i].plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--')
    axes[i].set_xlim([-0.02, 1.02])
    axes[i].set_ylim([-0.02, 1.02])
    axes[i].set_title(f'Test Class: {class_names[i]}', fontsize=12, fontweight='bold', color='#1a202c')
    axes[i].set_xlabel('False Positive Rate', fontsize=10)
    axes[i].set_ylabel('True Positive Rate', fontsize=10)
    axes[i].legend(loc="lower right", fontsize=9)
    axes[i].grid(True, linestyle=':', alpha=0.6)

for j in range(NUM_CLASSES, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Test Set Emotion Class ROC Performance Matrix', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()